In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv("C://Users//ravir//Downloads//employee_it_tickets_8000.csv")

In [3]:
df.head()

,ticket_id,short_description,description,category,subcategory,priority,assignment_group,resolution
0,INC02216,Problem: I need application access,The employee reports this issue during a clien...,Access,System Access,Medium,Access Management,Access rights were corrected.
1,INC02583,Please help: Remote VPN access is unavailable ...,This problem is preventing the employee from c...,VPN,VPN Authentication,High,Network Support,Remote access settings were checked.
2,INC01663,Request: Remote VPN access is unavailable,The employee reports this issue for the last f...,VPN,VPN Connection,Medium,Network Support,VPN credentials were refreshed.
3,INC03028,Problem: Possible malware detected,The employee reports this issue on my company ...,Security,Malware,High,Security Operations,Security credentials were reset.
4,INC04344,Problem: I cannot receive emails,New company emails are not appearing in the ma...,Email,Sending,High,Email Support,Email profile was recreated.


In [4]:
df.shape

(8000, 8)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ticket_id          8000 non-null   object
 1   short_description  8000 non-null   object
 2   description        8000 non-null   object
 3   category           8000 non-null   object
 4   subcategory        8000 non-null   object
 5   priority           8000 non-null   object
 6   assignment_group   8000 non-null   object
 7   resolution         8000 non-null   object
dtypes: object(8)
memory usage: 500.1+ KB


In [6]:
df.isnull().sum()

ticket_id            0
short_description    0
description          0
category             0
subcategory          0
priority             0
assignment_group     0
resolution           0
dtype: int64

In [7]:
df.duplicated().sum()

0

In [8]:
df["ticket_id"].duplicated().sum()

0

In [9]:
df["category"].value_counts()

category
Access      800
VPN         800
Security    800
Email       800
Hardware    800
Account     800
Network     800
Printer     800
Software    800
Database    800
Name: count, dtype: int64

In [10]:
df["category"].unique()

array(['Access', 'VPN', 'Security', 'Email', 'Hardware', 'Account',
       'Network', 'Printer', 'Software', 'Database'], dtype=object)

In [11]:
(df["short_description"].str.strip()=="").sum()

0

In [12]:
(df["description"].str.strip()=="").sum()

0

In [13]:
df["text"]=(df["short_description"].str.strip()+""+df["description"].str.strip()
           )
df[["short_description", "description","text"]].head()

,short_description,description,text
0,Problem: I need application access,The employee reports this issue during a clien...,Problem: I need application accessThe employee...
1,Please help: Remote VPN access is unavailable ...,This problem is preventing the employee from c...,Please help: Remote VPN access is unavailable ...
2,Request: Remote VPN access is unavailable,The employee reports this issue for the last f...,Request: Remote VPN access is unavailableThe e...
3,Problem: Possible malware detected,The employee reports this issue on my company ...,Problem: Possible malware detectedThe employee...
4,Problem: I cannot receive emails,New company emails are not appearing in the ma...,Problem: I cannot receive emailsNew company em...


In [14]:
X=df["text"]
y=df["category"]

print("X shape:",X.shape)
print("y shape:", y.shape)

X shape: (8000,)
y shape: (8000,)


In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=y
                                                    )
print("Training data:",X_train.shape)
print("Testing data:",X_test.shape)
                                                    

Training data: (6400,)
Testing data: (1600,)


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    min_df=2
)

X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

print("Training TF-IDF shape:",X_train_tfidf.shape)
print("Testing TF-IDF shape:",X_test_tfidf.shape)


Training TF-IDF shape: (6400, 1778)
Testing TF-IDF shape: (1600, 1778)


In [17]:
tfidf.fit_transform(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 198865 stored elements and shape (6400, 1778)>

In [18]:
tfidf.transform(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 50028 stored elements and shape (1600, 1778)>

In [19]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [20]:
y_pred=model.predict(X_test_tfidf)
print(y_pred[:20])

['Software' 'Network' 'VPN' 'Email' 'Access' 'Access' 'Access' 'Email'
 'Account' 'Printer' 'Printer' 'Email' 'Email' 'Access' 'Database'
 'Software' 'Software' 'Access' 'Network' 'Security']


In [21]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)

print("Accuracy:",accuracy)
print("Accuracy %:",accuracy*100)

Accuracy: 1.0
Accuracy %: 100.0


In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

      Access       1.00      1.00      1.00       160
     Account       1.00      1.00      1.00       160
    Database       1.00      1.00      1.00       160
       Email       1.00      1.00      1.00       160
    Hardware       1.00      1.00      1.00       160
     Network       1.00      1.00      1.00       160
     Printer       1.00      1.00      1.00       160
    Security       1.00      1.00      1.00       160
    Software       1.00      1.00      1.00       160
         VPN       1.00      1.00      1.00       160

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600



In [23]:
ticket=["VPN is not connecting from my home"]

ticket_tfidf=tfidf.transform(ticket)
prediction = model.predict(ticket_tfidf)
print("predicted category:",prediction[0])

predicted category: VPN


In [24]:
probalities=model.predict_proba(ticket_tfidf)
confidence=probalities.max()

print("Predicted Category:",prediction[0])
print("Confidence:",confidence*100,"%")

Predicted Category: VPN
Confidence: 95.56237985800678 %


In [25]:
import joblib
joblib.dump(model,"ticket_classifier.pkl")
joblib.dump(tfidf,"tfidf_vectorizer.pkl")

print("model and TF-IDF vectorizer saved")

model and TF-IDF vectorizer saved


In [26]:
from sklearn.linear_model import LogisticRegression
import joblib

# Subcategory target
y_subcategory_train = df.loc[X_train.index, "subcategory"]

# Create and train model
subcategory_model = LogisticRegression(max_iter=1000)

subcategory_model.fit(
    X_train_tfidf,
    y_subcategory_train
)

print("Subcategory model trained successfully!")

Subcategory model trained successfully!


In [27]:
y_priority_train = df.loc[X_train.index, "priority"]

priority_model = LogisticRegression(max_iter=1000)
priority_model.fit(X_train_tfidf, y_priority_train)

print("Priority trained")

Priority trained


In [28]:
y_assignment_train = df.loc[X_train.index, "assignment_group"]

assignment_model = LogisticRegression(max_iter=1000)
assignment_model.fit(X_train_tfidf, y_assignment_train)

print("Assignment Group trained")

Assignment Group trained


In [29]:
import joblib

joblib.dump(subcategory_model, "subcategory_classifier.pkl")
joblib.dump(priority_model, "priority_classifier.pkl")
joblib.dump(assignment_model, "assignment_group_classifier.pkl")

print("All 3 models saved successfully!")

All 3 models saved successfully!


In [30]:
# Test targets
y_subcategory_test = df.loc[X_test.index, "subcategory"]
y_priority_test = df.loc[X_test.index, "priority"]
y_assignment_test = df.loc[X_test.index, "assignment_group"]

print("Test targets created successfully!")

Test targets created successfully!


In [31]:
from sklearn.metrics import accuracy_score, classification_report

# ---------------- SUBCATEGORY ----------------
subcategory_pred = subcategory_model.predict(X_test_tfidf)

print("========== SUBCATEGORY ==========")
print("Accuracy:",
      round(accuracy_score(y_subcategory_test, subcategory_pred) * 100, 2), "%")


# ---------------- PRIORITY ----------------
priority_pred = priority_model.predict(X_test_tfidf)

print("\n========== PRIORITY ==========")
print("Accuracy:",
      round(accuracy_score(y_priority_test, priority_pred) * 100, 2), "%")


# ---------------- ASSIGNMENT GROUP ----------------
assignment_pred = assignment_model.predict(X_test_tfidf)

print("\n========== ASSIGNMENT GROUP ==========")
print("Accuracy:",
      round(accuracy_score(y_assignment_test, assignment_pred) * 100, 2), "%")

========== SUBCATEGORY ==========
Accuracy: 25.12 %

========== PRIORITY ==========
Accuracy: 41.75 %

========== ASSIGNMENT GROUP ==========
Accuracy: 100.0 %


In [32]:
from sklearn.metrics import classification_report

print("========== SUBCATEGORY REPORT ==========")
print(classification_report(
    y_subcategory_test,
    subcategory_pred,
    zero_division=0
))

print("\n========== PRIORITY REPORT ==========")
print(classification_report(
    y_priority_test,
    priority_pred,
    zero_division=0
))

print("\n========== ASSIGNMENT GROUP REPORT ==========")
print(classification_report(
    y_assignment_test,
    assignment_pred,
    zero_division=0
))

========== SUBCATEGORY REPORT ==========
                    precision    recall  f1-score   support

      Account Lock       0.36      0.39      0.37        44
       Application       0.16      0.14      0.14        37
Application Access       0.29      0.22      0.25        49
           Battery       0.14      0.14      0.14        21
           Browser       0.09      0.07      0.08        41
        Connection       0.30      0.29      0.29        45
      Connectivity       0.19      0.13      0.16        46
   Database Access       0.34      0.29      0.32        34
           Desktop       0.15      0.13      0.14        31
            Driver       0.16      0.25      0.19        40
 Folder Permission       0.15      0.11      0.13        36
      Installation       0.28      0.16      0.21        43
          Internet       0.24      0.33      0.28        42
          Keyboard       0.17      0.10      0.13        29
               LAN       0.20      0.16      0.18        3

In [34]:
print("\nCategory + Subcategory Distribution:")
print(
    df.groupby(["category", "subcategory"])
      .size()
      .sort_values(ascending=False)
)


Category + Subcategory Distribution:
category  subcategory       
VPN       VPN Connection        275
          Remote Access         264
          VPN Authentication    261
Access    Shared Drive          231
Email     Outlook               229
Database  Performance           224
Printer   Driver                223
Software  Microsoft Office      215
Network   Internet              215
Printer   Printing              212
Account   Account Lock          211
Security  Phishing              206
Access    Application Access    206
Account   Password              204
Security  Security Incident     202
Email     Receiving             202
Network   LAN                   199
Security  Suspicious Email      198
Software  Installation          197
Database  Connection            197
Software  Application           195
Account   Login                 194
Security  Malware               194
Network   WiFi                  194
Printer   Scanner               193
Software  Browser               1

In [36]:
# Confidence Threshold

CONFIDENCE_THRESHOLD = 0.70

if category_confidence >= CONFIDENCE_THRESHOLD:
    status = "AUTO ASSIGN"
else:
    status = "HUMAN REVIEW REQUIRED"

print("========== FINAL DECISION ==========")
print("Category:", predicted_category)
print("Confidence:", round(category_confidence * 100, 2), "%")
print("Decision:", status)

========== FINAL DECISION ==========
Category: VPN
Confidence: 94.68 %
Decision: AUTO ASSIGN
